# PatchCamelyon 70%-CPV static PCA benchmark 


## Design contract

This is a copy of the primary PatchCamelyon observation-scale ARL benchmark. The experimental structure is retained, but PCA dimensionality is selected separately for every seed and feature representation as the smallest number of Phase-I components reaching 70% cumulative explained variance. EWMA uses only the prespecified smoothing value 0.20. The default drift grid uses one severity level, 1.00. To include other or multiple levels, edit only `severity_levels` in `Config`, for example `(0.50, 1.00)`.

Outputs are isolated below `outputs/cpv70_ewma_primary/patchcamelyon/static_pca`. Dedicated per-seed and aggregate PCA-dimension files record the number of components required by every feature representation.


In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import math
import os
import pickle
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view
from PIL import Image
from scipy import special, stats as sstats
from sklearn.decomposition import PCA
from torchvision import datasets, models
from torchvision.transforms import functional as TF

%matplotlib inline
warnings.filterwarnings("ignore")

QUICK = os.environ.get("DRIFT_QUICK", "0") == "1"
RUN_FULL = os.environ.get("DRIFT_RUN_FULL", "1") == "1"
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)


@dataclass
class Config:
    dataset_name: str = "patchcamelyon"
    seeds: Tuple[int, ...] = (42,) if QUICK else tuple(range(42, 62))
    split_master_seed: int = 20260730
    stream_master_seed: int = 20260731
    split_ratios: Tuple[float, float, float] = (0.50, 0.25, 0.25)
    split_scheme: str = "seed_repartitioned_wsi_grouped_50_25_25"

    window: int = 50
    stride: int = 1
    changepoint: int = 50
    total_horizon: int = 1000
    target_arl0_observations: float = 370.0
    arl0_tol: float = 0.10
    arl0_cal_episodes: int = 20 if QUICK else 1000
    arl0_test_episodes: int = 20 if QUICK else 1000
    arl0_horizon_mult: int = 5
    arl0_bootstrap_block_steps: int = 100
    arl0_ci_bootstrap_reps: int = 100 if QUICK else 2000
    mc_arl1_reps: int = 2 if QUICK else 20

    ic_class: int = 0
    ooc_class: int = 1
    ooc_rate_max: float = 0.50
    # Default: one severity. Add values here, e.g. (0.50, 1.00).
    severity_levels: Tuple[float, ...] = (1.00,)
    patterns: Tuple[str, ...] = ("sudden", "incremental", "gradual")
    mechanisms: Tuple[str, ...] = ("jpeg", "stretch", "saturation", "ooc")
    jpeg_q_hi: int = 95
    jpeg_q_lo: int = 10
    stretch_max: float = 0.50
    saturation_max: float = 1.50

    # The float passed to sklearn PCA selects the smallest number of
    # components whose cumulative training-IC variance reaches 70%.
    pca_retention: float = 0.70
    pca_dim: float = 0.70  # compatibility key: target CPV, not k
    pca_dim_grid: Tuple[float, ...] = (0.70,)
    pca_fit_n: int = 50000
    ref_n: int = 1000
    hist_bins: int = 10
    mmd_ref_subsample: int = 500
    cusum_k: float = 0.5  # retained only because shared score helpers test it
    ewma_lambda: float = 0.20
    # Single prespecified EWMA value; it is not tuned on Phase-II results.
    ewma_lambda_grid: Tuple[float, ...] = (0.20,)

    extract_size: int = 96
    extract_batch: int = 256
    vae_latent: int = 64
    vae_epochs: int = 2 if QUICK else 20
    vae_batch: int = 64
    vae_eval_batch: int = 128

    data_root: str = "data"
    out_root: str = "outputs/cpv70_ewma_primary/patchcamelyon/static_pca"
    legacy_out_roots: Tuple[str, ...] = ("outputs/patchcamelyon_static_pca_legacy", "outputs/patchcamelyon_static_pca_legacy")
    force_recompute: bool = False
    pcam_download: bool = True
    wsi_split_trials: int = 100 if QUICK else 5000

    @property
    def postchange_horizon(self) -> int:
        return self.total_horizon - self.changepoint + 1

    @property
    def ramp_length(self) -> int:
        return int(round(self.postchange_horizon / 3))

    @property
    def target_arl0_score_steps(self) -> int:
        return 1 + int(math.ceil(
            (self.target_arl0_observations - self.window) / self.stride
        ))

    @property
    def target_arl0_steps(self) -> float:
        # Compatibility alias used by the validated shared detector helpers.
        return float(self.target_arl0_score_steps)

    @property
    def cooldown(self) -> int:
        return max(1, math.ceil(self.window / self.stride))

    @property
    def arl0_horizon_steps(self) -> int:
        return self.arl0_horizon_mult * self.target_arl0_score_steps


CFG = Config()
_project_base = Path.home() / "Documents" / "Stats Thesis notebooks and results"
_default_base = (
    _project_base if _project_base.is_dir()
    else (Path("/content") if Path("/content").is_dir() else Path.cwd())
)
RUN_BASE = Path(os.environ.get("DRIFT_RUN_BASE", str(_default_base))).expanduser()
RUN_BASE.mkdir(parents=True, exist_ok=True)
_existing_data = Path.home() / "Documents" / "Thesis Scripts finalized" / "data"
_default_data = _existing_data if _existing_data.is_dir() else RUN_BASE / "data"
CFG.data_root = os.environ.get("DRIFT_DATA_ROOT", str(_default_data))
CFG.out_root = str((RUN_BASE / Path(CFG.out_root)).absolute())
CFG.legacy_out_roots = ()  # never reuse thresholds or episode results from fixed-k runs
CFG.vae_legacy_out_roots = (str(RUN_BASE / 'outputs/patchcamelyon_static_pca_benchmark'),)
Path(CFG.out_root, "aggregate").mkdir(parents=True, exist_ok=True)

# Live progress is shown in the cell output and persisted for external monitoring.
_RUN_STARTED = time.time()
_PROGRESS_PATH = Path(CFG.out_root) / "progress.json"


def _format_duration(seconds):
    if seconds is None or not np.isfinite(seconds):
        return "--"
    seconds = max(0, int(round(seconds)))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:d}:{minutes:02d}:{seconds:02d}" if hours else f"{minutes:02d}:{seconds:02d}"


def report_progress(stage, seed=None, condition_index=None, condition_total=None,
                    episode=None, episode_total=None, condition_elapsed=None,
                    condition_eta=None, note="", live_line=False):
    seed_index = list(CFG.seeds).index(seed) + 1 if seed in CFG.seeds else None
    total_conditions = len(CFG.patterns) * len(CFG.mechanisms) * len(CFG.severity_levels)
    arl1_percent = None
    if seed_index and condition_index and episode is not None:
        completed = ((seed_index - 1) * total_conditions + condition_index - 1) * CFG.mc_arl1_reps + episode
        total = len(CFG.seeds) * total_conditions * CFG.mc_arl1_reps
        arl1_percent = 100.0 * completed / total
    payload = {
        "dataset": CFG.dataset_name, "stage": stage, "seed": seed,
        "seed_index": seed_index, "seed_total": len(CFG.seeds),
        "condition_index": condition_index, "condition_total": condition_total,
        "episode": episode, "episode_total": episode_total,
        "arl1_percent": arl1_percent, "note": note,
        "elapsed_seconds": time.time() - _RUN_STARTED,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    temporary = _PROGRESS_PATH.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    temporary.replace(_PROGRESS_PATH)
    parts = [f"[{payload['updated_at']}] {CFG.dataset_name}: {stage}"]
    if seed_index:
        parts.append(f"seed {seed_index}/{len(CFG.seeds)} ({seed})")
    if condition_index:
        parts.append(f"condition {condition_index}/{condition_total}")
    if episode is not None:
        parts.append(f"episode {episode}/{episode_total}")
    if arl1_percent is not None:
        parts.append(f"ARL1 grid {arl1_percent:5.1f}%")
    if condition_elapsed is not None:
        parts.append(f"condition elapsed {_format_duration(condition_elapsed)}")
    if condition_eta is not None:
        parts.append(f"ETA {_format_duration(condition_eta)}")
    if note:
        parts.append(note)
    print(" | ".join(parts), end="\r" if live_line else "\n", flush=True)

BLOCKS = ("layer1", "layer2", "layer3", "layer4")
FEATURE_BLOCKS = BLOCKS + ("vae_latent",)
FIXED_METHODS = (
    "KSWIN", "KL-Gauss", "KL-Hist", "MMD", "HDDDM",
    "T2", "SPE", "Frechet",
)
METHODS = FIXED_METHODS + ("EWMA", "VAE")
PATTERNS = CFG.patterns
TRANSFORMS = ("jpeg", "stretch", "saturation")

assert CFG.window == CFG.changepoint
assert CFG.stride == 1
assert CFG.postchange_horizon == 951
assert CFG.ramp_length == 317
assert CFG.target_arl0_score_steps == 321
assert "CUSUM" not in METHODS and "CUSUM-R" not in METHODS

print(f"dataset={CFG.dataset_name}; device={DEVICE}; QUICK={QUICK}; RUN_FULL={RUN_FULL}")
print(
    f"first score=observation {CFG.window}; first drift=observation "
    f"{CFG.changepoint}; delay(first-score alarm)=1 observation"
)
print(
    f"ARL0 target={CFG.target_arl0_observations:g} observations "
    f"({CFG.target_arl0_score_steps} score evaluations); "
    f"ARL1 horizon={CFG.total_horizon}, maximum delay={CFG.postchange_horizon}"
)
print(
    f"{CFG.mc_arl1_reps} nested Monte Carlo episodes/condition/seed; "
    f"ramp={CFG.ramp_length}; severities={CFG.severity_levels}"
)


## Dataset adapter and seed-specific partitions


In [ ]:
ACTIVE_SEED = None
PARTITION_IDS = {}
TRAIN_IC_IDS = VALIDATION_IC_IDS = TEST_IC_IDS = TEST_OOC_IDS = TRAIN_ALL_IDS = None
SPLIT_HASH = None


def activate_partition(
    seed: int,
    parts: Dict[str, np.ndarray],
    cfg: Config,
    wsi_manifest: Optional[pd.DataFrame] = None,
):
    global ACTIVE_SEED, PARTITION_IDS, TRAIN_IC_IDS, VALIDATION_IC_IDS
    global TEST_IC_IDS, TEST_OOC_IDS, TRAIN_ALL_IDS, SPLIT_HASH
    required = {"train", "validation", "test"}
    if set(parts) != required:
        raise ValueError(f"Partition roles must be {required}")
    parts = {k: np.asarray(v, dtype=np.int64) for k, v in parts.items()}
    joined = np.concatenate([parts[k] for k in ("train", "validation", "test")])
    if len(np.unique(joined)) != len(LABELS) or set(joined) != set(range(len(LABELS))):
        raise RuntimeError("Seed partition is not disjoint and exhaustive")
    PARTITION_IDS = parts
    TRAIN_ALL_IDS = parts["train"]
    TRAIN_IC_IDS = TRAIN_ALL_IDS[LABELS[TRAIN_ALL_IDS] == cfg.ic_class]
    VALIDATION_IC_IDS = parts["validation"][LABELS[parts["validation"]] == cfg.ic_class]
    TEST_IC_IDS = parts["test"][LABELS[parts["test"]] == cfg.ic_class]
    TEST_OOC_IDS = parts["test"][LABELS[parts["test"]] != cfg.ic_class]
    if min(len(TEST_IC_IDS), len(TEST_OOC_IDS)) < cfg.total_horizon:
        raise RuntimeError("Held-out IC/OOC pool is too small for unique 1,000-observation episodes")
    digest = hashlib.sha256()
    for role in ("train", "validation", "test"):
        digest.update(role.encode())
        digest.update(np.sort(parts[role]).tobytes())
    SPLIT_HASH = digest.hexdigest()[:16]
    ACTIVE_SEED = seed

    split_dir = Path(cfg.out_root) / "splits" / f"seed{seed}_{SPLIT_HASH}"
    split_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for role, ids in parts.items():
        np.save(split_dir / f"{role}_ids.npy", ids)
        rows.append({
            "seed": seed, "split_hash": SPLIT_HASH, "partition": role,
            "images": len(ids),
            "ic": int((LABELS[ids] == cfg.ic_class).sum()),
            "ooc": int((LABELS[ids] != cfg.ic_class).sum()),
        })
    pd.DataFrame(rows).to_csv(split_dir / "summary.csv", index=False)
    if wsi_manifest is not None:
        wsi_manifest.to_csv(split_dir / "wsi_manifest.csv", index=False)
        role_sets = [
            set(wsi_manifest.loc[wsi_manifest.partition == role, "wsi"])
            for role in ("train", "validation", "test")
        ]
        assert not (role_sets[0] & role_sets[1] or role_sets[0] & role_sets[2] or role_sets[1] & role_sets[2])
    print(pd.DataFrame(rows).to_string(index=False))
    return SPLIT_HASH


def partition_pools():
    if ACTIVE_SEED is None:
        raise RuntimeError("Call build_seed_partition(seed, CFG) first")
    return TRAIN_IC_IDS, VALIDATION_IC_IDS, TEST_IC_IDS, TEST_OOC_IDS


In [ ]:
import h5py
from torchvision.datasets.utils import check_integrity, download_file_from_google_drive

_IN_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
_IN_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def to_unit_tensor(u8: np.ndarray) -> torch.Tensor:
    x = torch.from_numpy(np.ascontiguousarray(u8)).to(DEVICE)
    return x.permute(0, 3, 1, 2).contiguous().float().div_(255.0)


def to_backbone_input(u8: np.ndarray) -> torch.Tensor:
    x = to_unit_tensor(u8)
    if x.shape[-2:] != (CFG.extract_size, CFG.extract_size):
        x = F.interpolate(x, size=CFG.extract_size, mode="bilinear", align_corners=False)
    return (x - _IN_MEAN) / _IN_STD


def augment_with_codes(u8: np.ndarray, codes: np.ndarray) -> np.ndarray:
    out = np.ascontiguousarray(u8.copy())
    codes = np.asarray(codes, dtype=np.uint8)
    flip = codes >= 4
    out[flip] = out[flip][:, :, ::-1, :]
    rotations = codes % 4
    for k in (1, 2, 3):
        mask = rotations == k
        if mask.any():
            out[mask] = np.rot90(out[mask], k=k, axes=(1, 2))
    return np.ascontiguousarray(out)


def augment_batch(u8: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return augment_with_codes(u8, rng.integers(0, 8, size=len(u8), dtype=np.uint8))


def _stable_u64(values: np.ndarray, seed: int) -> np.ndarray:
    x = np.asarray(values, dtype=np.uint64) + np.uint64(seed) + np.uint64(0x9E3779B97F4A7C15)
    x = (x ^ (x >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
    x = (x ^ (x >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
    return x ^ (x >> np.uint64(31))


def augment_test_batch(u8: np.ndarray, positions: np.ndarray, seed: int) -> np.ndarray:
    return augment_with_codes(u8, (_stable_u64(positions, seed) & np.uint64(7)).astype(np.uint8))


_PCAM_FILES = {
    "train": ("camelyonpatch_level_2_split_train_x.h5", "camelyonpatch_level_2_split_train_y.h5"),
    "val": ("camelyonpatch_level_2_split_valid_x.h5", "camelyonpatch_level_2_split_valid_y.h5"),
    "test": ("camelyonpatch_level_2_split_test_x.h5", "camelyonpatch_level_2_split_test_y.h5"),
}
_PCAM_META_FILES = {
    "train": ("camelyonpatch_level_2_split_train_meta.csv", "1XoaGG3ek26YLFvGzmkKeOz54INW0fruR", "5a3dd671e465cfd74b5b822125e65b0a"),
    "val": ("camelyonpatch_level_2_split_valid_meta.csv", "16hJfGFCZEcvR3lr38v3XCaD5iH1Bnclg", "67589e00a4a37ec317f2d1932c7502ca"),
    "test": ("camelyonpatch_level_2_split_test_meta.csv", "19tj7fBlQQrd4DapCjhZrom_fA4QlHqN4", "3455fd69135b66734e1008f3af684566"),
}

for _split in ("train", "val", "test"):
    datasets.PCAM(CFG.data_root, split=_split, download=CFG.pcam_download)
_pcam_base = Path(CFG.data_root) / "pcam"
for _split, (_name, _file_id, _md5) in _PCAM_META_FILES.items():
    _path = _pcam_base / _name
    if not check_integrity(str(_path), _md5):
        if not CFG.pcam_download:
            raise FileNotFoundError(_path)
        download_file_from_google_drive(_file_id, str(_pcam_base), filename=_name, md5=_md5)


class LazyPCAMCorpus:
    split_order = ("train", "val", "test")

    def __init__(self, root: str):
        self.base = Path(root) / "pcam"
        self._x_handles = {}
        labels, self.offsets, self.split_ids = [], {}, {}
        offset = 0
        for split in self.split_order:
            x_name, y_name = _PCAM_FILES[split]
            with h5py.File(self.base / y_name, "r") as handle:
                y = np.asarray(handle["y"][:, 0, 0, 0], dtype=np.int64)
            with h5py.File(self.base / x_name, "r") as handle:
                n, h, w, channels = handle["x"].shape
            if (h, w, channels) != (96, 96, 3) or len(y) != n:
                raise ValueError(f"Unexpected PCam array shape in {split}")
            self.offsets[split] = (offset, offset + n)
            self.split_ids[split] = np.arange(offset, offset + n, dtype=np.int64)
            labels.append(y)
            offset += n
        self.labels = np.concatenate(labels)
        self.shape = (offset, 96, 96, 3)

    def _images(self, split):
        if split not in self._x_handles:
            self._x_handles[split] = h5py.File(self.base / _PCAM_FILES[split][0], "r")
        return self._x_handles[split]["x"]

    def __getitem__(self, key):
        scalar = np.isscalar(key)
        ids = np.atleast_1d(np.asarray(key, dtype=np.int64))
        out = np.empty((len(ids), 96, 96, 3), dtype=np.uint8)
        for split in self.split_order:
            lo, hi = self.offsets[split]
            pos = np.flatnonzero((ids >= lo) & (ids < hi))
            if len(pos):
                local = ids[pos] - lo
                unique, inverse = np.unique(local, return_inverse=True)
                out[pos] = self._images(split)[unique][inverse]
        return out[0] if scalar else out


IMAGES = LazyPCAMCorpus(CFG.data_root)
LABELS = IMAGES.labels


def load_pcam_metadata() -> pd.DataFrame:
    parts = []
    for split in IMAGES.split_order:
        name, _, md5 = _PCAM_META_FILES[split]
        path = IMAGES.base / name
        if not check_integrity(str(path), md5):
            raise RuntimeError(f"PCam metadata integrity failure: {path}")
        frame = pd.read_csv(path, index_col=0).reset_index(drop=True)
        lo, hi = IMAGES.offsets[split]
        frame["global_id"] = np.arange(lo, hi, dtype=np.int64)
        frame["orig_split"] = split
        parts.append(frame)
    meta = pd.concat(parts, ignore_index=True)
    meta["label"] = LABELS[meta.global_id.to_numpy()]
    if meta.groupby("wsi").orig_split.nunique().max() != 1:
        raise ValueError("A WSI occurs in multiple original storage splits")
    return meta


PCAM_META = load_pcam_metadata()


def build_seed_partition(seed: int, cfg: Config):
    part_names = np.asarray(["train", "validation", "test"])
    ratios = np.asarray(cfg.split_ratios)
    stats = (
        PCAM_META.groupby(["orig_split", "wsi"], as_index=False)
        .agg(
            n0=("label", lambda x: int((x == cfg.ic_class).sum())),
            n1=("label", lambda x: int((x == cfg.ooc_class).sum())),
        )
    )
    stats["positive_wsi"] = stats.n1 > 0
    totals = stats[["n0", "n1"]].sum().to_numpy(float)
    targets = ratios[:, None] * totals[None, :]
    rng = np.random.default_rng(np.random.SeedSequence([cfg.split_master_seed, seed]))
    strata = list(stats.groupby(["orig_split", "positive_wsi"]).groups.values())
    best = None
    for trial in range(cfg.wsi_split_trials):
        assignment = np.empty(len(stats), dtype=np.int8)
        for raw_indices in strata:
            indices = np.asarray(list(raw_indices), dtype=np.int64)
            order = rng.permutation(indices)
            raw = ratios * len(indices)
            counts = np.floor(raw).astype(int)
            remainder = len(indices) - counts.sum()
            if remainder:
                counts[np.argsort(-(raw - counts))[:remainder]] += 1
            start = 0
            for part, count in enumerate(counts):
                assignment[order[start:start + count]] = part
                start += count
        counts = np.stack([
            stats.loc[assignment == part, ["n0", "n1"]].sum().to_numpy(float)
            for part in range(3)
        ])
        loss = float(np.sum(((counts - targets) / np.maximum(targets, 1)) ** 2))
        if counts[2].min() < cfg.total_horizon:
            loss += 1e6
        if best is None or loss < best[0]:
            best = (loss, assignment.copy(), trial)
    stats["partition"] = part_names[best[1]]
    mapping = stats.set_index("wsi").partition.to_dict()
    assigned = PCAM_META.wsi.map(mapping)
    parts = {
        part: PCAM_META.loc[assigned == part, "global_id"].to_numpy(np.int64)
        for part in part_names
    }
    activate_partition(seed, parts, cfg, wsi_manifest=stats)
    return parts


def build_backbone() -> nn.Module:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model = model.to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


## Frozen ImageNet ResNet-18 feature probes


In [ ]:
class BlockProbeExtractor:
    """Frozen ResNet-18 probes: global average and maximum pooling per stage."""

    def __init__(self, model: nn.Module):
        self.model = model.to(DEVICE).eval()
        self._buf: Dict[str, torch.Tensor] = {}
        for name in BLOCKS:
            getattr(self.model, name).register_forward_hook(self._make_hook(name))

    def _make_hook(self, name):
        def hook(_module, _inputs, output):
            self._buf[name] = torch.cat(
                [output.mean(dim=(2, 3)), output.amax(dim=(2, 3))], dim=1
            )
        return hook

    @torch.no_grad()
    def extract(self, u8: np.ndarray, batch: Optional[int] = None):
        batch = batch or CFG.extract_batch
        output = {name: [] for name in BLOCKS}
        for start in range(0, len(u8), batch):
            self._buf.clear()
            _ = self.model(to_backbone_input(u8[start:start + batch]))
            for name in BLOCKS:
                output[name].append(self._buf[name].float().cpu().numpy())
        return {name: np.concatenate(values) for name, values in output.items()}


EXTRACTOR = BlockProbeExtractor(build_backbone())
_first_parts = build_seed_partition(CFG.seeds[0], CFG)
_probe = EXTRACTOR.extract(IMAGES[TRAIN_IC_IDS[:8]])
assert {name: value.shape[1] for name, value in _probe.items()} == {
    "layer1": 128, "layer2": 256, "layer3": 512, "layer4": 1024,
}
print("ResNet-18 probe dimensions validated")


## Drift transformations


In [ ]:
def jpeg_batch(
    u8: np.ndarray, quality: int
) -> np.ndarray:
    out = np.empty_like(u8)
    for i, img in enumerate(u8):
        buf = io.BytesIO()
        Image.fromarray(img).save(
            buf, format="JPEG", quality=int(quality)
        )
        buf.seek(0)
        out[i] = np.asarray(Image.open(buf).convert("RGB"))
    return out


def stretch_batch(u8: np.ndarray, factor: float) -> np.ndarray:
    """Horizontal resize followed by a centred crop to the original shape."""
    height, original_width = u8.shape[1:3]
    stretched_width = int(round(original_width * factor))
    if stretched_width <= original_width:
        return u8
    tensor = (
        torch.from_numpy(np.ascontiguousarray(u8))
        .permute(0, 3, 1, 2)
        .float()
    )
    tensor = F.interpolate(
        tensor,
        size=(height, stretched_width),
        mode="bilinear",
        align_corners=False,
    )
    left = (stretched_width - original_width) // 2
    tensor = tensor[:, :, :, left:left + original_width]
    return (
        tensor.round()
        .clamp(0, 255)
        .byte()
        .permute(0, 2, 3, 1)
        .numpy()
    )


def saturation_batch(u8: np.ndarray, factor: float) -> np.ndarray:
    tensor = (
        torch.from_numpy(np.ascontiguousarray(u8))
        .permute(0, 3, 1, 2)
        .float()
        .div(255.0)
    )
    tensor = TF.adjust_saturation(tensor, factor)
    return (
        tensor.mul(255.0)
        .round()
        .clamp(0, 255)
        .byte()
        .permute(0, 2, 3, 1)
        .numpy()
    )


def apply_transform(
    name: str, u8: np.ndarray, s: float, cfg: Config
) -> np.ndarray:
    """Shared injection maps; s=0 is the untouched input.

    JPEG quality: 95 -> 10; horizontal scale: 1 -> 1.5 then centre-crop;
    saturation factor: 1 -> 2.5.
    """
    if s <= 0:
        return u8
    if name == "jpeg":
        quality = int(
            round(cfg.jpeg_q_hi - (cfg.jpeg_q_hi - cfg.jpeg_q_lo) * s)
        )
        return jpeg_batch(u8, quality)
    if name == "stretch":
        return stretch_batch(u8, 1.0 + cfg.stretch_max * s)
    if name == "saturation":
        return saturation_batch(u8, 1.0 + cfg.saturation_max * s)
    raise ValueError(name)


## Detector statistics


In [ ]:
def ledoit_wolf_batched(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Vectorized Ledoit-Wolf shrinkage covariance for a batch of windows.
    x: (nw, m, d). Returns means (nw, d) and covariances (nw, d, d).
    Matches sklearn.covariance.LedoitWolf (normalization by m)."""
    nw, m, d = x.shape
    mu = x.mean(axis=1)
    xc = x - mu[:, None, :]
    s = xc.transpose(0, 2, 1) @ xc / m       # batched BLAS, = einsum nmi,nmj->nij
    tr = np.einsum("nii->n", s)
    mu_i = tr / d
    s_frob2 = np.einsum("nij,nij->n", s, s)
    d2 = s_frob2 - 2.0 * mu_i * tr + d * mu_i ** 2
    sq_norms = np.einsum("nmi,nmi->nm", xc, xc)
    b2 = np.minimum((np.mean(sq_norms ** 2, axis=1) - s_frob2) / m, d2)
    shrink = np.where(d2 > 1e-30, b2 / np.maximum(d2, 1e-30), 1.0)
    cov = (1.0 - shrink)[:, None, None] * s
    cov[:, np.arange(d), np.arange(d)] += (shrink * mu_i)[:, None]
    return mu, cov


def prepare_reference(ref: np.ndarray, cfg: Config) -> Dict:
    """Precompute everything the five methods need about the fixed IC
    reference sample. Deterministic given `ref` (the RBF bandwidth uses the
    first mmd_ref_subsample rows), so it is exactly reproducible at eval time."""
    ref = np.asarray(ref, dtype=np.float64)
    r, d = ref.shape
    mu_r, cov_r = ledoit_wolf_batched(ref[None])
    mu_r, cov_r = mu_r[0], cov_r[0]
    cov_r_inv = np.linalg.inv(cov_r)
    _, logdet_r = np.linalg.slogdet(cov_r)
    # symmetric PSD square root of the reference covariance (for Frechet)
    evals_r, evecs_r = np.linalg.eigh(cov_r)
    cov_r_sqrt = (evecs_r * np.sqrt(np.clip(evals_r, 0.0, None))) @ evecs_r.T
    # histogram bins: reference-decile edges, open-ended outer bins
    qs = np.linspace(0, 1, cfg.hist_bins + 1)[1:-1]
    edges = np.quantile(ref, qs, axis=0).T                # (d, bins-1)
    counts = np.empty((d, cfg.hist_bins))
    for j in range(d):
        ids = np.searchsorted(edges[j], ref[:, j], side="right")
        counts[j] = np.bincount(ids, minlength=cfg.hist_bins)
    p_raw = counts / r
    p_lap = (counts + 1.0) / (r + cfg.hist_bins)
    # RBF bandwidth: median heuristic on a fixed subsample
    sub = ref[:min(cfg.mmd_ref_subsample, r)]
    sq = ((sub[:, None, :] - sub[None, :, :]) ** 2).sum(-1)
    med = np.median(sq[np.triu_indices_from(sq, k=1)])
    gamma = 1.0 / max(med, 1e-12)
    ref_sq = (ref ** 2).sum(1)
    d_rr = ref_sq[:, None] + ref_sq[None, :] - 2.0 * ref @ ref.T
    k_rr = np.exp(-gamma * np.maximum(d_rr, 0.0))
    term_rr = (k_rr.sum() - r) / (r * (r - 1))
    return {"ref": ref, "sorted": np.sort(ref, axis=0), "mu": mu_r,
            "cov_inv": cov_r_inv, "logdet": logdet_r,
            "cov_tr": float(np.trace(cov_r)), "cov_sqrt": cov_r_sqrt,
            "edges": edges,
            "p_raw": p_raw, "p_lap": p_lap, "gamma": gamma,
            "ref_sq": ref_sq, "term_rr": term_rr,
            "mu_marg": ref.mean(axis=0), "sd_marg": ref.std(axis=0) + 1e-12}


def pca_spe(pca: PCA, raw: np.ndarray, proj: np.ndarray) -> np.ndarray:
    """Per-sample squared prediction error (Q statistic) off the retained PCA
    basis: ||x_c||^2 - ||P x_c||^2, exact for orthonormal loadings (sklearn
    PCA), clipped at 0 against floating-point cancellation. `proj` is the
    already-computed projection so the basis is applied exactly once."""
    xc = np.asarray(raw, dtype=np.float64) - pca.mean_
    return np.maximum(np.einsum("nd,nd->n", xc, xc)
                      - np.einsum("nk,nk->n",
                                  np.asarray(proj, dtype=np.float64), proj),
                      0.0)


# ---- strided windowing ------------------------------------------------------
def n_windows_for(n_samples: int, cfg: Config) -> int:
    return (n_samples - cfg.window) // cfg.stride + 1


def make_windows(x: np.ndarray, w: int, stride: int) -> np.ndarray:
    """(n_samples, d) -> (nw, w, d); window i covers samples
    [i*stride, i*stride + w). stride == w gives disjoint tumbling windows."""
    v = sliding_window_view(x, w, axis=0)[::stride]      # (nw, d, w)
    return np.ascontiguousarray(v.transpose(0, 2, 1))


def window_means(values: np.ndarray, cfg: Config) -> np.ndarray:
    """Per-window mean of a scalar per-sample sequence on the strided grid
    (used by the VAE reconstruction-error detector)."""
    c = np.concatenate(([0.0], np.cumsum(values, dtype=np.float64)))
    starts = np.arange(n_windows_for(len(values), cfg)) * cfg.stride
    return (c[starts + cfg.window] - c[starts]) / cfg.window


def ks_d_batched(ref_sorted: np.ndarray, wins: np.ndarray) -> np.ndarray:
    """Exact two-sample KS statistic per (window, dim).
    ref_sorted: (R, d), sorted per column; wins: (nw, m, d).
    D = max( max_i((i+1)/m - F_ref(w_(i))), max_i(F_ref(w_(i)^-) - i/m) )."""
    nw, m, d = wins.shape
    r = ref_sorted.shape[0]
    ws = np.sort(wins, axis=1)
    i_hi = np.arange(1, m + 1) / m
    i_lo = np.arange(m) / m
    d_stat = np.empty((nw, d))
    for j in range(d):
        right = np.searchsorted(ref_sorted[:, j], ws[:, :, j], side="right") / r
        left = np.searchsorted(ref_sorted[:, j], ws[:, :, j], side="left") / r
        d_stat[:, j] = np.maximum((i_hi[None, :] - right).max(axis=1),
                                  (left - i_lo[None, :]).max(axis=1))
    return d_stat


def kswin_minp_stats(ref_prep: Dict, wins: np.ndarray) -> np.ndarray:
    """Per-PC KS p-values (asymptotic Kolmogorov distribution), combined via
    the minimum. One cutoff for this joint score is calibrated empirically;
    no separate Bonferroni alpha/d rule is applied."""
    r = ref_prep["sorted"].shape[0]
    m = wins.shape[1]
    en = math.sqrt(r * m / (r + m))
    return special.kolmogorov(en * ks_d_batched(ref_prep["sorted"], wins)).min(axis=1)


def gauss_moment_stats(ref_prep: Dict, wins: np.ndarray) -> Dict[str, np.ndarray]:
    """The three Gaussian-moment statistics from ONE shared Ledoit-Wolf pass
    over the windows:
    - KL-Gauss: closed-form KL( N_window || N_reference );
    - T2: Hotelling's T^2 of the window mean against the reference moments;
    - Frechet: closed-form Frechet/Wasserstein-2 distance between the two
      Gaussians (trace cross-term via batched eigendecomposition of the
      symmetric PSD matrix S_r^{1/2} S_w S_r^{1/2})."""
    nw, m, d = wins.shape
    mu_w, cov_w = ledoit_wolf_batched(np.asarray(wins, dtype=np.float64))
    diff = mu_w - ref_prep["mu"][None, :]
    maha = np.einsum("ni,ij,nj->n", diff, ref_prep["cov_inv"], diff)
    tr_term = np.einsum("ij,nji->n", ref_prep["cov_inv"], cov_w)
    _, logdet_w = np.linalg.slogdet(cov_w)
    kl = 0.5 * (tr_term + maha - d + ref_prep["logdet"] - logdet_w)
    t2 = m * maha
    s_half = ref_prep["cov_sqrt"]
    cross_ev = np.linalg.eigvalsh(s_half @ cov_w @ s_half)
    fd2 = (np.einsum("ni,ni->n", diff, diff)
           + np.einsum("nii->n", cov_w) + ref_prep["cov_tr"]
           - 2.0 * np.sqrt(np.clip(cross_ev, 0.0, None)).sum(axis=1))
    return {"KL-Gauss": kl, "T2": t2, "Frechet": np.maximum(fd2, 0.0)}


def window_bin_counts(edges: np.ndarray, wins: np.ndarray, n_bins: int) -> np.ndarray:
    """(nw, d, n_bins) histogram counts per window per dim, fixed ref-quantile
    edges with open-ended outer bins."""
    nw, m, d = wins.shape
    counts = np.empty((nw, d, n_bins))
    offsets = (np.arange(nw) * n_bins)[:, None]
    for j in range(d):
        ids = np.searchsorted(edges[j], wins[:, :, j], side="right")
        counts[:, j, :] = np.bincount((ids + offsets).ravel(),
                                      minlength=nw * n_bins).reshape(nw, n_bins)
    return counts


def mmd_stats(ref_prep: Dict, wins: np.ndarray, chunk: int = 200) -> np.ndarray:
    """Unbiased squared MMD, RBF kernel (median-heuristic bandwidth), between
    the fixed reference and each window."""
    ref, gamma = ref_prep["ref"], ref_prep["gamma"]
    ref_sq, term_rr = ref_prep["ref_sq"], ref_prep["term_rr"]
    nw, m, _ = wins.shape
    wins = np.asarray(wins, dtype=np.float64)
    out = np.empty(nw)
    for i in range(0, nw, chunk):
        wc = wins[i:i + chunk]
        w_sq = np.einsum("cmd,cmd->cm", wc, wc)
        d_ww = w_sq[:, :, None] + w_sq[:, None, :] - 2.0 * np.einsum("cmd,cnd->cmn", wc, wc)
        term_ww = (np.exp(-gamma * np.maximum(d_ww, 0.0)).sum(axis=(1, 2)) - m) / (m * (m - 1))
        d_rw = w_sq[:, :, None] + ref_sq[None, None, :] - 2.0 * np.einsum("cmd,rd->cmr", wc, ref)
        term_rw = np.exp(-gamma * np.maximum(d_rw, 0.0)).mean(axis=(1, 2))
        out[i:i + len(wc)] = term_rr + term_ww - 2.0 * term_rw
    return out


def mmd_stats_stream(ref_prep: Dict, proj: np.ndarray, cfg: Config,
                     chunk: int = 20000) -> np.ndarray:
    """Exact unbiased MMD^2 per window in O(n*R + n*m) via a per-sample
    decomposition (algebraically identical to mmd_stats — asserted below):
    the ref-window cross term is a rolling mean of per-sample kernel means
    phi_t = mean_r k(x_t, r), and the within-window pair sum is assembled
    from banded kernels k(x_t, x_{t+d}), d = 1..m-1, via cumulative sums.
    Required at small strides, where per-window evaluation would cost
    O(n_windows * m * R)."""
    ref, gamma = ref_prep["ref"], ref_prep["gamma"]
    ref_sq, term_rr = ref_prep["ref_sq"], ref_prep["term_rr"]
    x = np.asarray(proj, dtype=np.float64)
    n = len(x)
    m = cfg.window
    x_sq = np.einsum("nd,nd->n", x, x)
    phi = np.empty(n)
    for i in range(0, n, chunk):
        xc = x[i:i + chunk]
        d_rw = x_sq[i:i + chunk, None] + ref_sq[None, :] - 2.0 * xc @ ref.T
        phi[i:i + len(xc)] = np.exp(-gamma * np.maximum(d_rw, 0.0)).mean(axis=1)
    starts = np.arange(n_windows_for(n, cfg)) * cfg.stride
    cphi = np.concatenate(([0.0], np.cumsum(phi)))
    term_rw = (cphi[starts + m] - cphi[starts]) / m
    pair_sum = np.zeros(len(starts))
    for d in range(1, m):
        band = np.exp(-gamma * np.maximum(
            x_sq[:-d] + x_sq[d:] - 2.0 * np.einsum("nd,nd->n", x[:-d], x[d:]), 0.0))
        cb = np.concatenate(([0.0], np.cumsum(band)))
        pair_sum += cb[starts + m - d] - cb[starts]
    term_ww = 2.0 * pair_sum / (m * (m - 1))
    return term_rr + term_ww - 2.0 * term_rw


def _standardized_window_means(ref_prep: Dict, proj: np.ndarray,
                               cfg: Config) -> np.ndarray:
    """(nw, d) per-PC window means on the stride grid, standardized by the
    reference marginals: u = (mean - mu_ref) / (sd_ref / sqrt(m))."""
    x = np.asarray(proj, dtype=np.float64)
    c = np.vstack([np.zeros((1, x.shape[1])), np.cumsum(x, axis=0)])
    starts = np.arange(n_windows_for(len(x), cfg)) * cfg.stride
    wm = (c[starts + cfg.window] - c[starts]) / cfg.window
    return (wm - ref_prep["mu_marg"]) / (ref_prep["sd_marg"] / math.sqrt(cfg.window))


def cusum_stats_stream(ref_prep: Dict, proj: np.ndarray, cfg: Config) -> np.ndarray:
    """Two-sided per-PC CUSUM on standardized window means, allowance k,
    family-wise max over PCs. No in-chart reset: S+ is computed exactly via
    the prefix-min identity S+_t = P_t - min(0, min_{tau<=t} P_tau); alarm
    clustering is handled by the shared event-dedup cooldown."""
    u = _standardized_window_means(ref_prep, proj, cfg)
    zero = np.zeros((1, u.shape[1]))

    def one_sided(v):
        p = np.cumsum(v - cfg.cusum_k, axis=0)
        run_min = np.minimum.accumulate(np.vstack([zero, p]), axis=0)[1:]
        return p - np.minimum(run_min, 0.0)

    return np.maximum(one_sided(u), one_sided(-u)).max(axis=1)


def ewma_stats_stream(ref_prep: Dict, proj: np.ndarray, cfg: Config) -> np.ndarray:
    """Multivariate EWMA on standardized window means (z_0 = 0): statistic
    ||z_t||^2 * (2 - lambda) / lambda (T^2 with identity covariance — PCs are
    decorrelated under IC; residual miscalibration is absorbed by the
    empirical AIET search)."""
    from scipy.signal import lfilter
    u = _standardized_window_means(ref_prep, proj, cfg)
    lam = cfg.ewma_lambda
    z = lfilter([lam], [1.0, -(1.0 - lam)], u, axis=0)
    return (z ** 2).sum(axis=1) * (2.0 - lam) / lam


def _cusum_reset_scan(u: np.ndarray, cfg: Config, h: float,
                      max_events: Optional[int] = None,
                      chunk: int = 20000) -> Tuple[np.ndarray, int]:
    """Two-sided per-PC CUSUM WITH reset-on-alarm: all charts restart at zero
    after an alarm. Exact segment-wise computation via the prefix-minimum
    identity S+_t = P_t - min(P_{seg-1}, min_{seg<=s<=t} P_s), chunked so the
    total cost is O(n*d) per threshold candidate. Returns (alarm step
    indices, steps scanned) — scanning stops early after `max_events` alarms
    (used by the calibration search)."""
    k = cfg.cusum_k
    p_up = np.cumsum(u - k, axis=0)
    p_dn = np.cumsum(-u - k, axis=0)
    n, d = u.shape
    zeros = np.zeros(d)
    alarms: List[int] = []
    start = 0
    while start < n:
        cur_min_up = (p_up[start - 1] if start > 0 else zeros).copy()
        cur_min_dn = (p_dn[start - 1] if start > 0 else zeros).copy()
        pos, hit = start, None
        while pos < n:
            end = min(pos + chunk, n)
            pu, pd_ = p_up[pos:end], p_dn[pos:end]
            rm_u = np.minimum.accumulate(
                np.vstack([cur_min_up[None], pu]), axis=0)[1:]
            rm_d = np.minimum.accumulate(
                np.vstack([cur_min_dn[None], pd_]), axis=0)[1:]
            stat = np.maximum(pu - rm_u, pd_ - rm_d).max(axis=1)
            over = np.flatnonzero(stat > h)
            if len(over):
                hit = pos + int(over[0])
                break
            cur_min_up, cur_min_dn = rm_u[-1], rm_d[-1]
            pos = end
        if hit is None:
            break
        alarms.append(hit)
        if max_events is not None and len(alarms) >= max_events:
            return np.asarray(alarms, dtype=np.int64), hit + 1
        start = hit + 1
    return np.asarray(alarms, dtype=np.int64), n


def _dedup_events(indices: np.ndarray, cooldown: int) -> List[int]:
    events, nxt = [], -1
    for i in indices:
        if i >= nxt:
            events.append(int(i))
            nxt = i + cooldown
    return events


def calibrate_cusum_reset(u: np.ndarray, cfg: Config):
    """Bisection on the reset-CUSUM chart limit h until the calibration
    estimator (stream-origin-to-first-event plus successive completed gaps
    between cooldown-deduplicated events) lands in the ARL0 target band.
    The right-censored tail after the final event is not included. The reset
    makes the statistic sequence
    threshold-dependent, so the shared order-statistics search does not
    apply; each candidate h is evaluated by a full scan (with early stop)."""
    target, tol = cfg.target_arl0_steps, cfg.arl0_tol

    def arl_for(h: float):
        al, scanned = _cusum_reset_scan(u, cfg, h, max_events=400)
        ev = _dedup_events(al, cfg.cooldown)
        if not ev:
            return np.inf
        gaps = np.diff(np.concatenate(([-1], ev)))
        return float(gaps.mean())

    # upper bound: the no-reset statistic dominates the reset one
    p = np.cumsum(u - cfg.cusum_k, axis=0)
    hi = float((p - np.minimum(np.minimum.accumulate(p, axis=0), 0.0)).max())
    p = np.cumsum(-u - cfg.cusum_k, axis=0)
    hi = max(hi, float((p - np.minimum(np.minimum.accumulate(p, axis=0),
                                       0.0)).max())) + 1e-9
    lo, best = 0.0, None
    for _ in range(40):
        h = 0.5 * (lo + hi)
        arl = arl_for(h)
        score = (abs(math.log(arl / target))
                 if np.isfinite(arl) and arl > 0 else np.inf)
        if best is None or score < best[0]:
            best = (score, h, arl)
        if target * (1 - tol) <= arl <= target * (1 + tol):
            return h, arl, True
        if arl > target:
            hi = h          # too few alarms -> lower the limit
        else:
            lo = h
    return best[1], best[2], False


def cusum_reset_alarm_bool(u: np.ndarray, cfg: Config, h: float) -> np.ndarray:
    """Boolean alarm sequence of the calibrated reset-CUSUM on a stream."""
    al, _ = _cusum_reset_scan(u, cfg, h)
    out = np.zeros(len(u), dtype=bool)
    out[al] = True
    return out


def compute_window_stats(ref_prep: Dict, wins: np.ndarray, cfg: Config) -> Dict:
    """One statistic per window for the windowed PCA-feature methods (MMD is
    computed stream-wise, see stream_stats; SPE needs the raw feature space
    and has its own path). The histogram counts are computed once and shared
    by KL-Hist / Hellinger-fixed (legacy HDDDM label); the Gaussian
    moments once for KL-Gauss / T2 / Frechet. direction='high' alarms on stat
    > threshold; 'low' (KS-fixed, legacy KSWIN min-p label)
    on stat < threshold."""
    m = wins.shape[1]
    counts = window_bin_counts(ref_prep["edges"], wins, cfg.hist_bins)
    q_lap = (counts + 1.0) / (m + cfg.hist_bins)
    q_raw = counts / m
    p_lap = ref_prep["p_lap"][None]
    kl_hist = (q_lap * np.log(q_lap / p_lap)).sum(axis=2).mean(axis=1)
    bc = np.sqrt(q_raw * ref_prep["p_raw"][None]).sum(axis=2)
    hdddm = np.sqrt(np.clip(1.0 - bc, 0.0, None)).max(axis=1)
    gauss = gauss_moment_stats(ref_prep, wins)
    return {
        "KSWIN": (kswin_minp_stats(ref_prep, wins), "low"),
        "KL-Gauss": (gauss["KL-Gauss"], "high"),
        "KL-Hist": (kl_hist, "high"),
        "HDDDM": (hdddm, "high"),
        "T2": (gauss["T2"], "high"),
        "Frechet": (gauss["Frechet"], "high"),
    }


def stream_stats(ref_prep: Dict, proj: np.ndarray, cfg: Config,
                 chunk: int = 4000) -> Dict:
    """All projection-based method statistics over a whole stream on the
    strided window grid: the windowed methods chunked over windows to bound
    memory, MMD via the exact streaming decomposition, CUSUM/EWMA as
    sequential charts. (SPE is raw-space and handled by its callers.)"""
    n_w = n_windows_for(len(proj), cfg)
    acc: Optional[Dict] = None
    for i0 in range(0, n_w, chunk):
        i1 = min(i0 + chunk, n_w)
        seg = proj[i0 * cfg.stride:(i1 - 1) * cfg.stride + cfg.window]
        wins = make_windows(seg, cfg.window, cfg.stride)
        part = compute_window_stats(ref_prep, wins, cfg)
        if acc is None:
            acc = {m: ([v], d) for m, (v, d) in part.items()}
        else:
            for m, (v, d) in part.items():
                acc[m][0].append(v)
    out = {m: (np.concatenate(vs), d) for m, (vs, d) in acc.items()}
    out["MMD"] = (mmd_stats_stream(ref_prep, proj, cfg), "high")
    out["CUSUM"] = (cusum_stats_stream(ref_prep, proj, cfg), "high")
    out["EWMA"] = (ewma_stats_stream(ref_prep, proj, cfg), "high")
    return out


# ---- sanity checks --------------------------------------------------------
_rng = np.random.default_rng(0)
_ref = _rng.normal(size=(200, 3))
_wins = _rng.normal(size=(5, 30, 3))
_d_mine = ks_d_batched(np.sort(_ref, axis=0), _wins)
for _w in range(5):
    for _j in range(3):
        assert abs(_d_mine[_w, _j]
                   - sstats.ks_2samp(_ref[:, _j], _wins[_w, :, _j]).statistic) < 1e-12

from sklearn.covariance import LedoitWolf as _SkLW
_x = _rng.normal(size=(1, 50, 8)) * np.linspace(0.5, 3.0, 8)
_mu, _cov = ledoit_wolf_batched(_x)
_sk = _SkLW().fit(_x[0])
assert np.allclose(_cov[0], _sk.covariance_, atol=1e-10)
assert np.allclose(_mu[0], _sk.location_, atol=1e-12)

_seq = _rng.normal(size=(200, 2))
assert n_windows_for(200, CFG) == (200 - CFG.window) // CFG.stride + 1
_wv = make_windows(_seq, CFG.window, CFG.stride)
assert np.array_equal(_wv[1], _seq[CFG.stride:CFG.stride + CFG.window])

# streaming MMD must equal the direct per-window estimator exactly
_refp = prepare_reference(_rng.normal(size=(120, 4)), CFG)
_stream = _rng.normal(size=(400, 4))
_direct = mmd_stats(_refp, make_windows(_stream, CFG.window, CFG.stride))
assert np.allclose(mmd_stats_stream(_refp, _stream, CFG), _direct, atol=1e-10)

# vectorized CUSUM/EWMA must match their scalar recursions
_u = _standardized_window_means(_refp, _stream, CFG)
_sp, _ref_c = 0.0, []
for _v in _u[:, 0]:
    _sp = max(0.0, _sp + _v - CFG.cusum_k)
    _ref_c.append(_sp)
_p = np.cumsum(_u[:, 0] - CFG.cusum_k)
_s = _p - np.minimum(np.minimum.accumulate(_p), 0.0)
assert np.allclose(_s, _ref_c, atol=1e-12)
_z, _ref_e = 0.0, []
for _v in _u[:, 0]:
    _z = CFG.ewma_lambda * _v + (1 - CFG.ewma_lambda) * _z
    _ref_e.append(_z)
from scipy.signal import lfilter as _lf
assert np.allclose(_lf([CFG.ewma_lambda], [1.0, -(1.0 - CFG.ewma_lambda)],
                       _u[:, 0]), _ref_e, atol=1e-12)

# reset-on-alarm CUSUM scan must match its scalar recursion exactly
def _slow_reset_cusum(u, k, h):
    sp = np.zeros(u.shape[1])
    sn = np.zeros(u.shape[1])
    out = []
    for i, row in enumerate(u):
        sp = np.maximum(0.0, sp + row - k)
        sn = np.maximum(0.0, sn - row - k)
        if max(sp.max(), sn.max()) > h:
            out.append(i)
            sp[:] = 0.0
            sn[:] = 0.0
    return out

for _h in (2.0, 5.0, 9.0):
    _fast, _ = _cusum_reset_scan(_u, CFG, _h, chunk=37)   # odd chunk on purpose
    assert list(_fast) == _slow_reset_cusum(_u, CFG.cusum_k, _h)

# T2 / Frechet closed forms vs direct per-window computation
from scipy.linalg import sqrtm as _sqrtm
_wt = make_windows(_stream, CFG.window, CFG.stride)[:6]
_gs = gauss_moment_stats(_refp, _wt)
_muw, _covw = ledoit_wolf_batched(np.asarray(_wt, dtype=np.float64))
_covr = _refp["cov_sqrt"] @ _refp["cov_sqrt"]
for _i in range(len(_wt)):
    _d0 = _muw[_i] - _refp["mu"]
    assert abs(_gs["T2"][_i]
               - CFG.window * _d0 @ _refp["cov_inv"] @ _d0) < 1e-8
    _cross = _sqrtm(_refp["cov_sqrt"] @ _covw[_i] @ _refp["cov_sqrt"]).real
    _fd_ref = (_d0 @ _d0 + np.trace(_covw[_i]) + np.trace(_covr)
               - 2.0 * np.trace(_cross))
    assert abs(_gs["Frechet"][_i] - _fd_ref) < 1e-8
# Frechet of the reference against itself must vanish
_gs0 = gauss_moment_stats(_refp, _refp["ref"][None])
assert abs(_gs0["Frechet"][0]) < 1e-8

# SPE norm-difference identity vs explicit residual reconstruction
_pca_t = PCA(n_components=3, random_state=0).fit(_stream)
_pr_t = _pca_t.transform(_stream)
_res_t = _stream - _pca_t.mean_ - _pr_t @ _pca_t.components_
assert np.allclose(pca_spe(_pca_t, _stream, _pr_t),
                   (_res_t ** 2).sum(axis=1), atol=1e-10)

print("sanity checks passed (KS vs scipy; Ledoit-Wolf vs sklearn; windowing; "
      "streaming MMD vs direct; CUSUM/EWMA recursions; reset-CUSUM scan; "
      "T2/Frechet closed forms vs scipy.sqrtm; SPE residual identity)")


## Dataset-specific VAE


In [ ]:
class ConvVAE(nn.Module):
    """Convolutional VAE for native 96×96×3 PCam patches."""

    def __init__(self, latent: int = 64):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(inplace=True),      # 48×48
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(inplace=True),     # 24×24
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(inplace=True),    # 12×12
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(inplace=True),   # 6×6
            nn.Flatten(),
        )
        self.fc_mu = nn.Linear(256 * 6 * 6, latent)
        self.fc_lv = nn.Linear(256 * 6 * 6, latent)
        self.fc_dec = nn.Linear(latent, 256 * 6 * 6)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        return self.dec(self.fc_dec(z).view(-1, 256, 6, 6))

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.decode(z), mu, lv


@torch.no_grad()
def vae_recon_errors(model: nn.Module, u8: np.ndarray,
                     batch: Optional[int] = None) -> np.ndarray:
    """Per-sample pixel reconstruction MSE with deterministic encoding
    (decode the posterior mean)."""
    batch = batch or CFG.vae_eval_batch
    outs = []
    for i in range(0, len(u8), batch):
        x = to_unit_tensor(u8[i:i + batch])
        mu, _ = model.encode(x)
        recon = model.decode(mu)
        outs.append(((recon - x) ** 2).mean(dim=(1, 2, 3)).cpu().numpy())
    return np.concatenate(outs).astype(np.float64)


@torch.no_grad()
def vae_latents(model: nn.Module, u8: np.ndarray,
                batch: Optional[int] = None) -> np.ndarray:
    """Posterior-mean latent vectors mu(x): the VAE encoder used as a feature
    extractor (block 'vae_latent'), fed to the same PCA + divergence pipeline
    as the ResNet probes."""
    batch = batch or CFG.vae_eval_batch
    outs = []
    for i in range(0, len(u8), batch):
        mu, _ = model.encode(to_unit_tensor(u8[i:i + batch]))
        outs.append(mu.float().cpu().numpy())
    return np.concatenate(outs)


In [ ]:
def split_metadata(seed: int, cfg: Config):
    return {
        "dataset": cfg.dataset_name,
        "seed": int(seed),
        "split_hash": SPLIT_HASH,
        "latent": cfg.vae_latent,
        "epochs": cfg.vae_epochs,
        "architecture": type(ConvVAE(cfg.vae_latent)).__name__,
        "preprocessing": "dataset_specific_primary",
    }


def train_seed_vae(seed: int, cfg: Config) -> nn.Module:
    if ACTIVE_SEED != seed:
        build_seed_partition(seed, cfg)
    metadata = split_metadata(seed, cfg)
    outdir = Path(cfg.out_root) / "vae_models"
    outdir.mkdir(parents=True, exist_ok=True)
    path = outdir / f"seed{seed}_{SPLIT_HASH}_vae_e{cfg.vae_epochs}_d{cfg.vae_latent}.pt"
    legacy_paths = [
        Path(root) / "vae_models"
        / f"seed{seed}_{SPLIT_HASH}_vae_e{cfg.vae_epochs}_d{cfg.vae_latent}.pt"
        for root in cfg.vae_legacy_out_roots
    ]
    candidates = [path] if cfg.force_recompute else [path, *legacy_paths]
    for candidate in candidates:
        if not candidate.exists():
            continue
        payload = torch.load(candidate, map_location="cpu")
        if isinstance(payload, dict) and payload.get("metadata") == metadata:
            if candidate != path:
                torch.save(payload, path)
            model = ConvVAE(cfg.vae_latent)
            model.load_state_dict(payload["state_dict"])
            print(f"[seed {seed}] loaded compatible VAE {candidate}")
            return model.to(DEVICE).eval()
        print(f"[seed {seed}] rejected incompatible VAE cache {candidate}")

    rng = np.random.default_rng(np.random.SeedSequence([cfg.stream_master_seed, seed, 10]))
    torch.manual_seed(seed)
    model = ConvVAE(cfg.vae_latent).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(cfg.vae_epochs):
        order = rng.permutation(TRAIN_IC_IDS)
        model.train()
        total = 0.0
        for start in range(0, len(order), cfg.vae_batch):
            ids = order[start:start + cfg.vae_batch]
            x = to_unit_tensor(augment_batch(IMAGES[ids], rng))
            recon, mu, logvar = model(x)
            reconstruction = F.mse_loss(recon, x, reduction="sum") / len(ids)
            kl = -0.5 * torch.sum(1 + logvar - mu.square() - logvar.exp()) / len(ids)
            loss = reconstruction + kl
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total += float(loss) * len(ids)
        if epoch == 0 or (epoch + 1) % 5 == 0 or epoch + 1 == cfg.vae_epochs:
            print(f"[seed {seed}] VAE epoch {epoch + 1}/{cfg.vae_epochs}: {total / len(order):.3f}")
    model.eval()
    torch.save({"metadata": metadata, "state_dict": model.state_dict()}, path)
    return model


## Observation-scale first-alarm calibration helpers


In [ ]:
def proper_steps_to_observations(steps, cfg: Config):
    steps = np.asarray(steps, dtype=np.float64)
    return cfg.window + (steps - 1.0) * cfg.stride


def moving_block_plan(n_steps, n_episodes, horizon, block_steps, rng):
    if n_steps < block_steps:
        raise ValueError("Calibration score bank is shorter than one bootstrap block")
    n_blocks = int(math.ceil(horizon / block_steps))
    return rng.integers(0, n_steps - block_steps + 1, size=(n_episodes, n_blocks))


def _records_from_values(values, direction, plan, horizon, block_steps):
    values = np.asarray(values, dtype=np.float64)
    if direction == "low":
        values = -values
    records = []
    for starts in plan:
        indices = np.concatenate([
            np.arange(start, start + block_steps) for start in starts
        ])[:horizon]
        running = np.maximum.accumulate(values[indices])
        changed = np.flatnonzero(running > np.r_[-np.inf, running[:-1]])
        records.append((running[changed], changed + 1))
    return records


def _ewma_values(u, lam):
    from scipy.signal import lfilter
    z = lfilter([lam], [1.0, -(1.0 - lam)], np.asarray(u), axis=0)
    return np.square(z).sum(axis=1) * (2.0 - lam) / lam


def _ewma_records(u, lam, plan, horizon, block_steps):
    from scipy.signal import lfilter
    records = []
    dimension = u.shape[1]
    for starts in plan:
        state = np.zeros((1, dimension))
        running_max, elapsed = -np.inf, 0
        rec_values, rec_times = [], []
        for start in starts:
            take = min(block_steps, horizon - elapsed)
            if take <= 0:
                break
            z, state = lfilter(
                [lam], [1.0, -(1.0 - lam)], u[start:start + take],
                axis=0, zi=state,
            )
            values = np.square(z).sum(axis=1) * (2.0 - lam) / lam
            running = np.maximum.accumulate(np.maximum(values, running_max))
            changed = np.flatnonzero(running > np.r_[running_max, running[:-1]])
            rec_values.extend(running[changed])
            rec_times.extend(elapsed + changed + 1)
            running_max = float(running[-1])
            elapsed += take
        records.append((np.asarray(rec_values), np.asarray(rec_times)))
    return records


def _run_lengths(records, threshold, horizon):
    lengths = np.full(len(records), horizon, dtype=np.int64)
    observed = np.zeros(len(records), dtype=bool)
    for episode, (values, times) in enumerate(records):
        hit = int(np.searchsorted(values, threshold, side="right"))
        if hit < len(values):
            lengths[episode] = times[hit]
            observed[episode] = True
    return lengths, observed


def calibrate_records(records, direction, cfg: Config):
    candidates = np.unique(np.concatenate([v for v, _ in records if len(v)]))
    target = cfg.target_arl0_score_steps
    horizon = cfg.arl0_horizon_steps
    lo, hi, best = 0, len(candidates) - 1, None
    while lo <= hi:
        mid = (lo + hi) // 2
        limit = float(candidates[mid])
        lengths, observed = _run_lengths(records, limit, horizon)
        mean_steps = float(lengths.mean())
        candidate = (abs(math.log(mean_steps / target)), limit, lengths, observed)
        if best is None or candidate[0] < best[0]:
            best = candidate
        if mean_steps < target:
            lo = mid + 1
        else:
            hi = mid - 1
    _, transformed, lengths, observed = best
    threshold = transformed if direction == "high" else -transformed
    observations = proper_steps_to_observations(lengths, cfg)
    achieved = float(observations.mean())
    return {
        "threshold": float(threshold), "direction": direction,
        "cal_arl0_observations": achieved,
        "cal_arl0_score_steps": float(lengths.mean()),
        "cal_censor_rate": float((~observed).mean()),
        "cal_episodes": len(records),
        "in_band": bool(
            cfg.target_arl0_observations * (1 - cfg.arl0_tol)
            <= achieved
            <= cfg.target_arl0_observations * (1 + cfg.arl0_tol)
        ),
    }


def first_alarm(scores, threshold, direction):
    alarms = scores > threshold if direction == "high" else scores < threshold
    hits = np.flatnonzero(alarms)
    return None if not len(hits) else int(hits[0])


assert proper_steps_to_observations([1], CFG)[0] == 50
assert proper_steps_to_observations([321], CFG)[0] == 370
assert (CFG.window - CFG.changepoint + 1) == 1
print("Observation-scale ARL timing checks passed")


## Seed-specific fitting and threshold calibration


In [ ]:
def _extract_features(ids, rng, extractor, vae, augment=True):
    ids = np.asarray(ids, dtype=np.int64)
    images = IMAGES[ids]
    if augment:
        images = augment_batch(images, rng)
    features = extractor.extract(images)
    features["vae_latent"] = vae_latents(vae, images)
    return images, features


def _fit_pca(raw, target_cpv):
    """Covariance PCA fitted on training IC data only.

    sklearn's float n_components selects the smallest k whose cumulative
    explained-variance ratio is at least target_cpv. The full solver is
    required for the fractional rule. No feature-wise standardisation is
    introduced, so this differs from the primary benchmark only in component selection.
    """
    if not (0.0 < float(target_cpv) < 1.0):
        raise ValueError("target_cpv must be a proportion strictly between 0 and 1")
    if min(len(raw), raw.shape[1]) < 2:
        raise ValueError(f"Cannot fit PCA to {raw.shape}")
    return PCA(n_components=float(target_cpv), svd_solver="full").fit(raw)


def _pca_audit_fields(state, target_cpv, block):
    if block == "pixel":
        return {"pca_target_cpv": np.nan, "pca_dim": np.nan, "pca_cpv": np.nan}
    model = state["pca_models"][target_cpv][block]
    return {
        "pca_target_cpv": float(target_cpv),
        "pca_dim": int(model.n_components_),
        "pca_cpv": float(model.explained_variance_ratio_.sum()),
    }


def _pca_dimension_rows(state, seed, split_hash, dataset):
    rows = []
    for target_cpv, block_models in state["pca_models"].items():
        for block, model in block_models.items():
            rows.append({
                "dataset": dataset,
                "seed": int(seed),
                "split_hash": split_hash,
                "feature_representation": block,
                "phase1_fit_observations": int(len(state["fit_ids"])),
                "pca_target_cpv": float(target_cpv),
                "principal_components": int(model.n_components_),
                "achieved_variance_retention": float(
                    model.explained_variance_ratio_.sum()
                ),
            })
    return rows


def _write_pca_dimension_audit(state, seed, split_hash, dataset, outdir):
    frame = pd.DataFrame(
        _pca_dimension_rows(state, seed, split_hash, dataset)
    ).sort_values(["pca_target_cpv", "feature_representation"])
    frame.to_csv(Path(outdir) / "pca_dimensions.csv", index=False)
    return frame


def _score_bundle(raw_features, vae_errors, state, k, cfg: Config):
    bundle = {}
    for block in FEATURE_BLOCKS:
        pca = state["pca_models"][k][block]
        projected = pca.transform(raw_features[block]).astype(np.float64)
        reference = prepare_reference(state["ref_proj"][k][block], cfg)
        for method, (values, direction) in stream_stats(reference, projected, cfg).items():
            if method in FIXED_METHODS:
                bundle[(block, method, None)] = (values, direction)
        spe = window_means(pca_spe(pca, raw_features[block], projected), cfg)
        bundle[(block, "SPE", None)] = (spe, "high")
        u = _standardized_window_means(reference, projected, cfg)
        for lam in cfg.ewma_lambda_grid:
            bundle[(block, "EWMA", float(lam))] = (_ewma_values(u, lam), "high")
    if vae_errors is not None:
        bundle[("pixel", "VAE", None)] = (window_means(vae_errors, cfg), "high")
    return bundle


def fit_and_calibrate_seed(seed: int, cfg: Config, extractor):
    build_seed_partition(seed, cfg)
    outdir = Path(cfg.out_root) / "imagenet" / f"seed{seed}_{SPLIT_HASH}"
    outdir.mkdir(parents=True, exist_ok=True)
    state_path = outdir / "seed_state.pkl"
    metadata = {
        "version": "static_pca_cpv70_ewma020_severity1", "dataset": cfg.dataset_name,
        "seed": seed, "split_hash": SPLIT_HASH,
        "target_arl0_observations": cfg.target_arl0_observations,
        "window": cfg.window, "stride": cfg.stride,
        "pca_target_cpv": cfg.pca_retention,
        "ewma_lambdas": tuple(cfg.ewma_lambda_grid),
        "methods": METHODS,
    }
    legacy_state_paths = [
        Path(root) / "imagenet"
        / f"seed{seed}_{SPLIT_HASH}" / "seed_state.pkl"
        for root in cfg.legacy_out_roots
    ]
    state_candidates = [state_path] if cfg.force_recompute else [
        state_path, *legacy_state_paths
    ]
    for candidate in state_candidates:
        if not candidate.exists():
            continue
        with candidate.open("rb") as handle:
            state = pickle.load(handle)
        if state.get("metadata") == metadata:
            state["outdir"] = str(outdir)
            if candidate != state_path:
                with state_path.open("wb") as handle:
                    pickle.dump(state, handle)
            threshold_rows = [
                {"seed": seed, "split_hash": SPLIT_HASH,
                 **_pca_audit_fields(state, key[0], key[1]),
                 "block": key[1], "method": key[2], "parameter": key[3], **value}
                for key, value in state["thresholds"].items()
            ]
            pd.DataFrame(threshold_rows).to_csv(
                outdir / "thresholds.csv", index=False
            )
            _write_pca_dimension_audit(
                state, seed, SPLIT_HASH, cfg.dataset_name, outdir
            )
            print(f"[seed {seed}] loaded compatible fitted state {candidate}")
            return state

    vae = train_seed_vae(seed, cfg)
    rng = np.random.default_rng(np.random.SeedSequence([cfg.stream_master_seed, seed, 20]))
    fit_ids = rng.permutation(TRAIN_IC_IDS)[:min(cfg.pca_fit_n, len(TRAIN_IC_IDS))]
    _, fit_features = _extract_features(fit_ids, rng, extractor, vae, augment=True)
    ref_ids = rng.choice(TRAIN_IC_IDS, size=cfg.ref_n, replace=False)
    _, ref_features = _extract_features(ref_ids, rng, extractor, vae, augment=True)

    pca_models, ref_proj = {}, {}
    for k in cfg.pca_dim_grid:
        pca_models[k], ref_proj[k] = {}, {}
        for block in FEATURE_BLOCKS:
            pca_models[k][block] = _fit_pca(fit_features[block], k)
            ref_proj[k][block] = pca_models[k][block].transform(
                ref_features[block]
            ).astype(np.float64)

    validation_ids = rng.permutation(VALIDATION_IC_IDS)
    validation_images = augment_test_batch(
        IMAGES[validation_ids], np.arange(len(validation_ids)),
        cfg.stream_master_seed + seed * 1000 + 21,
    )
    validation_features = extractor.extract(validation_images)
    validation_features["vae_latent"] = vae_latents(vae, validation_images)
    validation_vae = vae_recon_errors(vae, validation_images)

    n_scores = len(validation_ids) - cfg.window + 1
    plan = moving_block_plan(
        n_scores, cfg.arl0_cal_episodes, cfg.arl0_horizon_steps,
        cfg.arl0_bootstrap_block_steps,
        np.random.default_rng(np.random.SeedSequence([cfg.stream_master_seed, seed, 22])),
    )
    state = {
        "metadata": metadata, "pca_models": pca_models, "ref_proj": ref_proj,
        "thresholds": {}, "fit_ids": fit_ids, "ref_ids": ref_ids,
        "validation_ids": validation_ids, "outdir": str(outdir),
    }
    # Save PCA dimensions before detector calibration so this design audit
    # survives even if a later calibration job is interrupted.
    _write_pca_dimension_audit(
        state, seed, SPLIT_HASH, cfg.dataset_name, outdir
    )
    for k in cfg.pca_dim_grid:
        bundle = _score_bundle(
            validation_features, validation_vae if k == cfg.pca_dim else None,
            state, k, cfg,
        )
        for (block, method, parameter), (values, direction) in bundle.items():
            if method == "EWMA":
                reference = prepare_reference(ref_proj[k][block], cfg)
                projected = pca_models[k][block].transform(validation_features[block])
                u = _standardized_window_means(reference, projected, cfg)
                records = _ewma_records(
                    u, parameter, plan, cfg.arl0_horizon_steps,
                    cfg.arl0_bootstrap_block_steps,
                )
            else:
                records = _records_from_values(
                    values, direction, plan, cfg.arl0_horizon_steps,
                    cfg.arl0_bootstrap_block_steps,
                )
            state["thresholds"][(k, block, method, parameter)] = calibrate_records(
                records, direction, cfg
            )
    with state_path.open("wb") as handle:
        pickle.dump(state, handle)
    threshold_rows = [
        {"seed": seed, "split_hash": SPLIT_HASH,
         **_pca_audit_fields(state, key[0], key[1]),
         "block": key[1], "method": key[2], "parameter": key[3], **value}
        for key, value in state["thresholds"].items()
    ]
    pd.DataFrame(threshold_rows).to_csv(outdir / "thresholds.csv", index=False)
    return state


## Paired, unique-identity Monte Carlo ARL₁ episodes


In [ ]:
def master_episode_manifest(seed: int, cfg: Config):
    if ACTIVE_SEED != seed:
        build_seed_partition(seed, cfg)
    outdir = Path(cfg.out_root) / "episode_manifests"
    outdir.mkdir(parents=True, exist_ok=True)
    path = outdir / f"seed{seed}_{SPLIT_HASH}_n{cfg.mc_arl1_reps}_h{cfg.total_horizon}.npz"
    if path.exists() and not cfg.force_recompute:
        data = np.load(path)
        return data["ic_orders"], data["ooc_orders"]
    ic_orders = np.empty((cfg.mc_arl1_reps, cfg.total_horizon), dtype=np.int64)
    ooc_orders = np.empty((cfg.mc_arl1_reps, cfg.total_horizon), dtype=np.int64)
    for replication in range(cfg.mc_arl1_reps):
        rng = np.random.default_rng(
            np.random.SeedSequence([cfg.stream_master_seed, seed, replication, 30])
        )
        ic_orders[replication] = rng.choice(
            TEST_IC_IDS, cfg.total_horizon, replace=False
        )
        ooc_orders[replication] = rng.choice(
            TEST_OOC_IDS, cfg.total_horizon, replace=False
        )
        assert len(np.unique(ic_orders[replication])) == cfg.total_horizon
        assert len(np.unique(ooc_orders[replication])) == cfg.total_horizon
    np.savez_compressed(path, ic_orders=ic_orders, ooc_orders=ooc_orders)
    return ic_orders, ooc_orders


def onset_strengths(pattern, severity, cfg: Config):
    j = np.arange(1, cfg.postchange_horizon + 1)
    if pattern == "sudden":
        return np.full_like(j, severity, dtype=float), np.ones_like(j, dtype=float)
    ramp = np.minimum(j / cfg.ramp_length, 1.0)
    if pattern == "incremental":
        return severity * ramp, ramp
    if pattern == "gradual":
        return np.full_like(j, severity, dtype=float), ramp
    raise ValueError(pattern)


def apply_transform_schedule(images, mechanism, strengths):
    output = images.copy()
    active = np.flatnonzero(strengths > 0)
    if not len(active):
        return output
    # JPEG uses integer quality, so grouping also avoids redundant encoder setup.
    if mechanism == "jpeg":
        qualities = np.rint(
            CFG.jpeg_q_hi - (CFG.jpeg_q_hi - CFG.jpeg_q_lo) * strengths
        ).astype(int)
        for quality in np.unique(qualities[active]):
            ids = active[qualities[active] == quality]
            output[ids] = jpeg_batch(output[ids], int(quality))
        return output
    # Stretch and saturation retain the exact source-notebook endpoint maps.
    for value in np.unique(strengths[active]):
        ids = active[np.isclose(strengths[active], value)]
        output[ids] = apply_transform(mechanism, output[ids], float(value), CFG)
    return output


def build_episode(seed, replication, pattern, mechanism, severity, ic_ids, ooc_ids, cfg):
    positions = np.arange(cfg.total_horizon, dtype=np.int64)
    images = augment_test_batch(
        IMAGES[ic_ids], positions,
        cfg.stream_master_seed + seed * 100000 + replication,
    )
    observed_ids = ic_ids.copy()
    strengths, ramp_probability = onset_strengths(pattern, severity, cfg)
    post = slice(cfg.changepoint - 1, cfg.total_horizon)
    rng = np.random.default_rng(np.random.SeedSequence([
        cfg.stream_master_seed, seed, replication,
        cfg.patterns.index(pattern), cfg.mechanisms.index(mechanism),
        int(round(severity * 100)), 31,
    ]))
    if mechanism != "ooc":
        if pattern == "gradual":
            mask = rng.random(cfg.postchange_horizon) < ramp_probability
            effective = strengths * mask
        else:
            effective = strengths
        images[post] = apply_transform_schedule(images[post], mechanism, effective)
    else:
        if pattern == "gradual":
            replace = (
                (rng.random(cfg.postchange_horizon) < ramp_probability)
                & (rng.random(cfg.postchange_horizon) < cfg.ooc_rate_max * severity)
            )
        else:
            replace = rng.random(cfg.postchange_horizon) < cfg.ooc_rate_max * strengths
        count = int(replace.sum())
        if count:
            replacement_ids = ooc_ids[:count]
            target_positions = np.flatnonzero(replace) + (cfg.changepoint - 1)
            images[target_positions] = augment_test_batch(
                IMAGES[replacement_ids], target_positions,
                cfg.stream_master_seed + seed * 100000 + replication + 1,
            )
            observed_ids[target_positions] = replacement_ids
    if len(np.unique(observed_ids)) != len(observed_ids):
        raise RuntimeError("Identity duplicated inside a Monte Carlo episode")
    return images, observed_ids


def detector_specs(cfg: Config):
    specs = []
    for block in FEATURE_BLOCKS:
        for method in FIXED_METHODS:
            specs.append(("primary", cfg.pca_dim, None, block, method))
        specs.append(("primary", cfg.pca_dim, cfg.ewma_lambda, block, "EWMA"))
    specs.append(("primary", cfg.pca_dim, None, "pixel", "VAE"))
    for k in cfg.pca_dim_grid:
        if k == cfg.pca_dim:
            continue
        for block in FEATURE_BLOCKS:
            for method in FIXED_METHODS:
                specs.append(("pca_dimension", k, None, block, method))
            specs.append(("pca_dimension", k, cfg.ewma_lambda, block, "EWMA"))
    for lam in cfg.ewma_lambda_grid:
        if np.isclose(lam, cfg.ewma_lambda):
            continue
        for block in FEATURE_BLOCKS:
            specs.append(("ewma_memory", cfg.pca_dim, lam, block, "EWMA"))
    return specs


def evaluate_episode(images, state, vae, cfg: Config):
    features = EXTRACTOR.extract(images)
    features["vae_latent"] = vae_latents(vae, images)
    vae_errors = vae_recon_errors(vae, images)
    bundles = {
        k: _score_bundle(features, vae_errors if k == cfg.pca_dim else None, state, k, cfg)
        for k in cfg.pca_dim_grid
    }
    rows = []
    for family, k, parameter, block, method in detector_specs(cfg):
        lookup_parameter = float(parameter) if method == "EWMA" else None
        values, direction = bundles[k][(block, method, lookup_parameter)]
        info = state["thresholds"][(k, block, method, lookup_parameter)]
        hit = first_alarm(values, info["threshold"], direction)
        detected = hit is not None
        delay = float(hit + 1) if detected else np.nan
        rows.append({
            "parameter_family": family, **_pca_audit_fields(state, k, block),
            "ewma_lambda": parameter if method == "EWMA" else np.nan,
            "block": block, "method": method,
            "detected": detected, "censored": not detected,
            "alarm_score_index": hit if detected else np.nan,
            "alarm_observation": cfg.changepoint + hit if detected else np.nan,
            "arl1_delay_observations": delay,
            "restricted_delay_observations": delay if detected else cfg.postchange_horizon,
            "cal_arl0_observations": info["cal_arl0_observations"],
        })
    return rows


def evaluate_seed(seed: int, cfg: Config, extractor):
    report_progress("seed setup", seed=seed, note="building split")
    build_seed_partition(seed, cfg)
    report_progress("phase-I calibration", seed=seed, note="PCA fit and ARL0 calibration")
    state = fit_and_calibrate_seed(seed, cfg, extractor)
    report_progress("VAE preparation", seed=seed, note="loading or training checkpoint")
    vae = train_seed_vae(seed, cfg)
    ic_orders, ooc_orders = master_episode_manifest(seed, cfg)
    seed_dir = Path(state["outdir"]) / "arl1_conditions"
    seed_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    condition_total = len(cfg.patterns) * len(cfg.mechanisms) * len(cfg.severity_levels)
    condition_index = 0
    for pattern in cfg.patterns:
        for mechanism in cfg.mechanisms:
            for severity in cfg.severity_levels:
                condition_index += 1
                condition_started = time.time()
                condition_note = f"{pattern}/{mechanism}/s={severity:g}"
                path = seed_dir / f"{pattern}_{mechanism}_s{severity:.2f}.csv"
                legacy_paths = [
                    Path(root) / "imagenet"
                    / f"seed{seed}_{SPLIT_HASH}" / "arl1_conditions"
                    / path.name
                    for root in cfg.legacy_out_roots
                ]
                loaded = False
                for candidate in (
                    [] if cfg.force_recompute else [path, *legacy_paths]
                ):
                    if not candidate.exists():
                        continue
                    cached = pd.read_csv(candidate)
                    required = set(range(cfg.mc_arl1_reps))
                    available = set(cached.replication.astype(int).unique())
                    if not required <= available:
                        continue
                    cached = cached[
                        cached.replication.astype(int) < cfg.mc_arl1_reps
                    ].copy()
                    cached.to_csv(path, index=False)
                    frames.append(cached)
                    print(
                        f"[seed {seed}] reused {cfg.mc_arl1_reps} episodes "
                        f"from {candidate.name}"
                    )
                    report_progress(
                        "ARL1 episodes", seed, condition_index, condition_total,
                        cfg.mc_arl1_reps, cfg.mc_arl1_reps, note=condition_note + " (reused)",
                    )
                    loaded = True
                    break
                if loaded:
                    continue
                rows = []
                for replication in range(cfg.mc_arl1_reps):
                    images, observed_ids = build_episode(
                        seed, replication, pattern, mechanism, severity,
                        ic_orders[replication], ooc_orders[replication], cfg,
                    )
                    episode_rows = evaluate_episode(images, state, vae, cfg)
                    for row in episode_rows:
                        row.update({
                            "dataset": cfg.dataset_name, "seed": seed,
                            "split_hash": SPLIT_HASH, "replication": replication,
                            "pattern": pattern, "mechanism": mechanism,
                            "severity": severity, "window": cfg.window,
                            "stride": cfg.stride, "changepoint": cfg.changepoint,
                            "total_horizon": cfg.total_horizon,
                            "unique_identities": len(np.unique(observed_ids)),
                        })
                    rows.extend(episode_rows)
                    completed_episodes = replication + 1
                    condition_elapsed = time.time() - condition_started
                    condition_eta = condition_elapsed / completed_episodes * (cfg.mc_arl1_reps - completed_episodes)
                    report_progress(
                        "ARL1 episodes", seed, condition_index, condition_total,
                        completed_episodes, cfg.mc_arl1_reps, condition_elapsed,
                        condition_eta, condition_note, live_line=completed_episodes < cfg.mc_arl1_reps,
                    )
                frame = pd.DataFrame(rows)
                frame.to_csv(path, index=False)
                frames.append(frame)
                print(f"[seed {seed}] completed {pattern}/{mechanism}/s={severity:g}")
    result = pd.concat(frames, ignore_index=True)
    result.to_csv(Path(state["outdir"]) / "arl1_episode_results.csv", index=False)
    return result


## Held-out ARL₀ verification


In [ ]:
def verify_heldout_arl0(seed: int, cfg: Config, extractor):
    build_seed_partition(seed, cfg)
    state = fit_and_calibrate_seed(seed, cfg, extractor)
    vae = train_seed_vae(seed, cfg)
    summary_path = Path(state["outdir"]) / "heldout_arl0_summary.csv"
    legacy_summary_paths = [
        Path(root) / "imagenet" / f"seed{seed}_{SPLIT_HASH}"
        / "heldout_arl0_summary.csv"
        for root in cfg.legacy_out_roots
    ]
    candidates = (
        [] if cfg.force_recompute
        else [summary_path, *legacy_summary_paths]
    )
    for candidate in candidates:
        if not candidate.exists():
            continue
        cached = pd.read_csv(candidate)
        compatible = (
            len(cached) == len(state["thresholds"])
            and set(cached.dataset.astype(str)) == {cfg.dataset_name}
            and set(cached.seed.astype(int)) == {seed}
            and set(cached.split_hash.astype(str)) == {SPLIT_HASH}
            and (cached.episodes.astype(int) >= cfg.arl0_test_episodes).all()
        )
        if compatible:
            cached.to_csv(summary_path, index=False)
            print(f"[seed {seed}] reused held-out ARL0 from {candidate}")
            return cached
    rng = np.random.default_rng(np.random.SeedSequence([cfg.stream_master_seed, seed, 40]))
    ids = rng.permutation(TEST_IC_IDS)
    images = augment_test_batch(
        IMAGES[ids], np.arange(len(ids)),
        cfg.stream_master_seed + seed * 1000 + 40,
    )
    features = extractor.extract(images)
    features["vae_latent"] = vae_latents(vae, images)
    vae_errors = vae_recon_errors(vae, images)
    n_scores = len(ids) - cfg.window + 1
    plan = moving_block_plan(
        n_scores, cfg.arl0_test_episodes, cfg.arl0_horizon_steps,
        cfg.arl0_bootstrap_block_steps,
        np.random.default_rng(np.random.SeedSequence([cfg.stream_master_seed, seed, 41])),
    )
    rows = []
    for k in cfg.pca_dim_grid:
        bundle = _score_bundle(features, vae_errors if k == cfg.pca_dim else None, state, k, cfg)
        for (block, method, parameter), (values, direction) in bundle.items():
            records = (
                _ewma_records(
                    _standardized_window_means(
                        prepare_reference(state["ref_proj"][k][block], cfg),
                        state["pca_models"][k][block].transform(features[block]), cfg,
                    ),
                    parameter, plan, cfg.arl0_horizon_steps,
                    cfg.arl0_bootstrap_block_steps,
                )
                if method == "EWMA"
                else _records_from_values(
                    values, direction, plan, cfg.arl0_horizon_steps,
                    cfg.arl0_bootstrap_block_steps,
                )
            )
            info = state["thresholds"][(k, block, method, parameter)]
            transformed_threshold = info["threshold"] if direction == "high" else -info["threshold"]
            lengths, observed = _run_lengths(records, transformed_threshold, cfg.arl0_horizon_steps)
            observations = proper_steps_to_observations(lengths, cfg)
            rows.append({
                "dataset": cfg.dataset_name, "seed": seed, "split_hash": SPLIT_HASH,
                **_pca_audit_fields(state, k, block), "block": block, "method": method,
                "ewma_lambda": parameter if method == "EWMA" else np.nan,
                "episodes": len(lengths), "events": int(observed.sum()),
                "censored": int((~observed).sum()),
                "censor_rate": float((~observed).mean()),
                "arl0_observations": float(observations.mean()),
                "arl0_score_steps": float(lengths.mean()),
                "target_arl0_observations": cfg.target_arl0_observations,
                "estimate_type": "empirical_mean" if observed.all() else "restricted_mean",
            })
    frame = pd.DataFrame(rows)
    frame.to_csv(summary_path, index=False)
    return frame


## Full execution and seed-first aggregation


In [ ]:
def aggregate_results(results: pd.DataFrame, arl0: pd.DataFrame, cfg: Config):
    aggregate = Path(cfg.out_root) / "aggregate"
    aggregate.mkdir(parents=True, exist_ok=True)
    results.to_csv(aggregate / "arl1_episode_results_all_seeds.csv", index=False)
    arl0.to_csv(aggregate / "heldout_arl0_all_seeds.csv", index=False)

    dimension_files = sorted(
        Path(cfg.out_root).glob("imagenet/seed*/pca_dimensions.csv")
    )
    if dimension_files:
        pca_dimensions = pd.concat(
            [pd.read_csv(path) for path in dimension_files], ignore_index=True
        ).sort_values(["seed", "feature_representation"])
        pca_dimensions.to_csv(
            aggregate / "pca_dimensions_all_seeds.csv", index=False
        )

    group = [
        "dataset", "parameter_family", "pca_target_cpv", "ewma_lambda",
        "block", "method", "pattern", "mechanism", "severity",
    ]
    seed_level = (
        results.groupby(group + ["seed"], dropna=False)
        .agg(
            episodes=("replication", "nunique"),
            pca_dim=("pca_dim", "first"),
            pca_cpv=("pca_cpv", "first"),
            detections=("detected", "sum"),
            detection_rate=("detected", "mean"),
            censor_rate=("censored", "mean"),
            mean_arl1_detected=("arl1_delay_observations", "mean"),
            median_arl1_detected=("arl1_delay_observations", "median"),
            restricted_mean_delay=("restricted_delay_observations", "mean"),
        )
        .reset_index()
    )
    seed_level["ordinary_arl1"] = np.where(
        seed_level.censor_rate == 0,
        seed_level.mean_arl1_detected,
        np.nan,
    )
    seed_level["restricted_mean_arl1_observations"] = (
        seed_level.restricted_mean_delay
    )
    seed_level.to_csv(aggregate / "arl1_seed_level.csv", index=False)
    seed_level.to_csv(
        aggregate / "restricted_arl1_seed_level_fully_deaggregated.csv",
        index=False,
    )
    across = (
        seed_level.groupby(group, dropna=False)
        .agg(
            seeds=("seed", "nunique"),
            mean_pca_dim=("pca_dim", "mean"),
            min_pca_dim=("pca_dim", "min"),
            max_pca_dim=("pca_dim", "max"),
            mean_pca_cpv=("pca_cpv", "mean"),
            mean_detection_rate=("detection_rate", "mean"),
            seed_sd_detection_rate=("detection_rate", "std"),
            mean_arl1_detected=("mean_arl1_detected", "mean"),
            seed_sd_arl1_detected=("mean_arl1_detected", "std"),
            mean_ordinary_arl1=("ordinary_arl1", "mean"),
            mean_restricted_delay=("restricted_mean_delay", "mean"),
            seed_sd_restricted_delay=("restricted_mean_delay", "std"),
            mean_censor_rate=("censor_rate", "mean"),
        )
        .reset_index()
    )
    critical = sstats.t.ppf(0.975, np.maximum(across.seeds - 1, 1))
    for mean_col, sd_col, prefix in (
        ("mean_detection_rate", "seed_sd_detection_rate", "detection_rate"),
        ("mean_arl1_detected", "seed_sd_arl1_detected", "arl1_detected"),
        ("mean_restricted_delay", "seed_sd_restricted_delay", "restricted_delay"),
    ):
        half = critical * across[sd_col] / np.sqrt(across.seeds)
        across[f"{prefix}_ci_lo"] = across[mean_col] - half
        across[f"{prefix}_ci_hi"] = across[mean_col] + half
    across["detection_rate_ci_lo"] = across.detection_rate_ci_lo.clip(lower=0)
    across["detection_rate_ci_hi"] = across.detection_rate_ci_hi.clip(upper=1)
    across["restricted_mean_arl1_observations"] = (
        across.mean_restricted_delay
    )
    across.to_csv(aggregate / "arl1_across_seeds.csv", index=False)
    across.to_csv(
        aggregate / "restricted_arl1_across_seeds_fully_deaggregated.csv",
        index=False,
    )

    # Machine-readable condition tables: no pooling of onset, mechanism,
    # severity, detector, representation block, or sensitivity parameter.
    deaggregated = aggregate / "deaggregated_restricted_arl1"
    deaggregated.mkdir(parents=True, exist_ok=True)
    for pattern in cfg.patterns:
        for mechanism in cfg.mechanisms:
            seed_slice = seed_level[
                (seed_level.pattern == pattern)
                & (seed_level.mechanism == mechanism)
            ].copy()
            across_slice = across[
                (across.pattern == pattern)
                & (across.mechanism == mechanism)
            ].copy()
            seed_slice.to_csv(
                deaggregated
                / f"{pattern}_{mechanism}_restricted_arl1_by_seed.csv",
                index=False,
            )
            across_slice.to_csv(
                deaggregated
                / f"{pattern}_{mechanism}_restricted_arl1_across_seeds.csv",
                index=False,
            )

    threshold_files = sorted(Path(cfg.out_root).glob("imagenet/seed*/thresholds.csv"))
    if threshold_files:
        calibration = pd.concat(
            [pd.read_csv(path) for path in threshold_files], ignore_index=True
        )
        calibration["relative_error"] = (
            calibration.cal_arl0_observations / cfg.target_arl0_observations - 1
        )
        calibration.to_csv(aggregate / "calibration_all_seeds.csv", index=False)
        audit = (
            calibration.groupby(["pca_target_cpv", "block", "method", "parameter"], dropna=False)
            .agg(
                seeds=("seed", "nunique"),
                mean_cal_arl0=("cal_arl0_observations", "mean"),
                max_abs_relative_error=("relative_error", lambda x: float(np.abs(x).max())),
                cells_outside_tolerance=("in_band", lambda x: int((~x.astype(bool)).sum())),
                mean_cal_censor_rate=("cal_censor_rate", "mean"),
            )
            .reset_index()
        )
        audit.to_csv(aggregate / "calibration_audit.csv", index=False)

    # Fully deaggregated plots: one representation block per figure and one
    # panel for each onset-by-mechanism cell. Curves retain detector identity.
    primary = across[across.parameter_family == "primary"].copy()
    for block in primary.block.unique():
        block_data = primary[primary.block == block]
        fig, axes = plt.subplots(
            len(cfg.patterns), len(cfg.mechanisms),
            figsize=(18, 12), sharex=True, sharey=True,
        )
        for row, pattern in enumerate(cfg.patterns):
            for column, mechanism in enumerate(cfg.mechanisms):
                axis = axes[row, column]
                panel = block_data[
                    (block_data.pattern == pattern)
                    & (block_data.mechanism == mechanism)
                ]
                for method, values in panel.groupby("method"):
                    curve = values.sort_values("severity")
                    axis.plot(
                        curve.severity,
                        curve.restricted_mean_arl1_observations,
                        marker="o", linewidth=1, label=method,
                    )
                axis.set_title(f"{pattern} / {mechanism}")
                axis.set_xlabel("severity")
                axis.set_ylabel("restricted mean ARL1 (observations)")
        axes[0, -1].legend(
            fontsize=6, bbox_to_anchor=(1.02, 1), loc="upper left"
        )
        fig.suptitle(
            f"{cfg.dataset_name}: deaggregated restricted ARL1 — {block}"
        )
        fig.tight_layout()
        fig.savefig(
            aggregate
            / f"restricted_arl1_deaggregated_{block}.png",
            dpi=160,
        )
        plt.close(fig)
    return seed_level, across


if RUN_FULL:
    _results, _arl0 = [], []
    report_progress("run started", note=f"{len(CFG.seeds)} seeds; {CFG.mc_arl1_reps} episodes per condition")
    for _seed in CFG.seeds:
        _results.append(evaluate_seed(_seed, CFG, EXTRACTOR))
        report_progress("held-out ARL0", seed=_seed, note="verification episodes")
        _arl0.append(verify_heldout_arl0(_seed, CFG, EXTRACTOR))
        report_progress("seed complete", seed=_seed)
    RESULTS = pd.concat(_results, ignore_index=True)
    ARL0 = pd.concat(_arl0, ignore_index=True)
    SEED_LEVEL, ACROSS_SEEDS = aggregate_results(RESULTS, ARL0, CFG)
    report_progress("complete", note=f"total elapsed {_format_duration(time.time() - _RUN_STARTED)}")
    display(ACROSS_SEEDS.head(30))
else:
    print("RUN_FULL=False: construction and helper validation complete; full fitting/evaluation skipped")


## Reproducibility and methodological audit


In [ ]:
# Static and lightweight methodological audit
assert CFG.window == 50 and CFG.changepoint == 50
assert np.isclose(CFG.pca_retention, 0.70) and CFG.pca_dim_grid == (0.70,)
assert CFG.total_horizon == 1000 and CFG.mc_arl1_reps in ({2} if QUICK else {20})
assert len(CFG.seeds) in ({1} if QUICK else {20})
assert CFG.ramp_length == 317
assert set(CFG.patterns) == {"sudden", "incremental", "gradual"}
assert "recurring" not in CFG.patterns
assert "CUSUM" not in METHODS and "CUSUM-R" not in METHODS
for severity in CFG.severity_levels:
    sudden, _ = onset_strengths("sudden", severity, CFG)
    incremental, _ = onset_strengths("incremental", severity, CFG)
    gradual, probability = onset_strengths("gradual", severity, CFG)
    assert sudden[0] == severity
    assert np.isclose(incremental[CFG.ramp_length - 1], severity)
    assert np.isclose(probability[CFG.ramp_length - 1], 1.0)
    assert np.allclose(gradual, severity)
assert len(detector_specs(CFG)) == len(set(detector_specs(CFG)))

_partition_hashes = []
for _seed in CFG.seeds:
    build_seed_partition(_seed, CFG)
    _partition_hashes.append(SPLIT_HASH)
    roles = [set(PARTITION_IDS[name]) for name in ("train", "validation", "test")]
    assert not (roles[0] & roles[1] or roles[0] & roles[2] or roles[1] & roles[2])
assert len(set(_partition_hashes)) == len(_partition_hashes), "Images must move between partitions between seeds"
build_seed_partition(CFG.seeds[0], CFG)

_ic, _ooc = master_episode_manifest(CFG.seeds[0], CFG)
assert len(np.unique(_ic[0])) == CFG.total_horizon
assert len(np.unique(_ooc[0])) == CFG.total_horizon
assert not set(_ic[0]) & set(_ooc[0])
print("benchmark audit passed: split movement, disjointness, timing, onset definitions, and unique test identities")

assert CFG.severity_levels == (1.00,)
assert CFG.ewma_lambda_grid == (0.20,)
